In [1]:
%%bash
set -uo pipefail
W=/kaggle/working
T=/tmp/repro
MAIN=$T/glassbox-mech
PY=$T/py/bin/python
REST='"numpy==1.26.4" "scipy==1.17.1" "einops==0.8.2" "transformer-lens==2.17.0" "transformers==4.57.6" "pyarrow==23.0.0" "datasets==4.5.0" "pandas==3.0.0" pytest'
rm -rf $T $W/reports && mkdir -p $T $W/reports
git clone -q https://github.com/designer-coderajay/glassbox-mech.git $MAIN
git -C $MAIN log -1 --format='%H %s' | tee $W/reports/main.commit.txt
pip install -q uv
uv venv $T/py --python 3.11 -q
uv pip install -q --python $PY --index-url https://download.pytorch.org/whl/cpu "torch==2.10.0"
eval uv pip install -q --python $PY $REST
uv pip install -q --python $PY -e $MAIN --no-deps
uv pip freeze --python $PY > $W/reports/env_freeze.txt
lscpu | head -20 | tee $W/reports/lscpu.txt
df -h /tmp $W | tee $W/reports/disk_start.txt
CPU="$(lscpu | grep 'Model name' | sed 's/Model name:[[:space:]]*//') / Linux x86-64 / Kaggle CPU"
prep () {
  git -C $MAIN worktree add -q $T/wt/$1 $2
  git -C $T/wt/$1 log -1 --format='%H %s' > $W/reports/$1.commit.txt
  ln -s $T/py $T/wt/$1/.venv-repro
  uv pip install -q --python $PY -e $T/wt/$1 --no-deps
}
cmp () {
  p=$1; shift
  uv pip install -q --python $PY -e $MAIN --no-deps
  mkdir -p $W/reports/$p.fresh
  for f in "$@"; do cp "$f" $W/reports/$p.fresh/$(echo "$f" | sed "s|$T/wt/$p/||; s|/|__|g") 2>/dev/null; done
  (cd $MAIN && $PY -m glassbox.v7.reproduce compare evidence/$p "$@" --reproducer "Ajay Pravin Mahale (author)" --environment "$CPU" --out $W/reports/$p.json) 2>&1 | tee $W/reports/$p.compare.txt
  git -C $MAIN worktree remove --force $T/wt/$p
  df -h /tmp $W | tail -2
}

prep v7-exp5-real-interventions e47cf4e
(cd $T/wt/v7-exp5-real-interventions && .venv-repro/bin/python experiments/v7/real_intervention/run.py --out repro/exp5) > $W/reports/v7-exp5-real-interventions.log 2>&1
cmp v7-exp5-real-interventions $T/wt/v7-exp5-real-interventions/repro/exp5/results.json

prep v7-exp6-cross-model 3314c58
(cd $T/wt/v7-exp6-cross-model && .venv-repro/bin/python experiments/v7/cross_model/run.py --out repro/exp6) > $W/reports/v7-exp6-cross-model.log 2>&1
cmp v7-exp6-cross-model $T/wt/v7-exp6-cross-model/repro/exp6/results.json

prep v7-exp3-noise-baseline 3cf24ac
(cd $T/wt/v7-exp3-noise-baseline && .venv-repro/bin/python experiments/v7/noise_baseline/part_a.py --out repro/exp3/part_a; .venv-repro/bin/python experiments/v7/noise_baseline/part_b.py --out repro/exp3/part_b) > $W/reports/v7-exp3-noise-baseline.log 2>&1
cmp v7-exp3-noise-baseline $T/wt/v7-exp3-noise-baseline/repro/exp3/part_a/results.json $T/wt/v7-exp3-noise-baseline/repro/exp3/part_b/results.json

df -h /tmp $W | tee $W/reports/disk_end.txt
cd $W && zip -qr repro_B.zip reports && ls -la reports && grep -h VERDICT reports/*.compare.txt

1bb823421f4ff790d5b53eb7333426718c96bebd prereg(v7): Exp 9 addendum 1, attempt 1 stopped by full disk; CPU torch, shared venv, /tmp worktrees for attempt 2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 76.2 MB/s eta 0:00:00
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  4
On-line CPU(s) list:                     0-3
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4399.99
Flags:   

Using Python 3.11.17 environment at: /tmp/repro/py
